# Módulo de Pré-processamento e Vetorização Binária para Bernoulli Naive Bayes
Disciplina de Probabilidade e Estatística (IFCE).

Neste notebook, abordaremos a **Seleção e Justificativa de Features** e o **Pré-processamento** de texto para o modelo de **Bernoulli Naive Bayes**, seguindo um experimento progressivo de refinamento do vocabulário.


### 1. Importando as Bibliotecas


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
import numpy as np
import pandas as pd
from scipy.sparse import spmatrix
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split

# Garante o download das stopwords em inglês do NLTK
nltk.download("stopwords", quiet=True)


True

### 2. Configurando os Caminhos do Projeto


In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "bbc-text.csv"

print(f"Raiz do Projeto: {PROJECT_ROOT}")
print(f"Caminho do CSV:  {RAW_CSV_PATH} (Existe? {RAW_CSV_PATH.exists()})")


Raiz do Projeto: C:\Users\diego\Desktop\repositorios\bayes-bbc-prediction
Caminho do CSV:  C:\Users\diego\Desktop\repositorios\bayes-bbc-prediction\data\raw\bbc-text.csv (Existe? True)


### 3. Carregamento dos Dados e Definição das Variáveis


In [3]:
df = pd.read_csv(RAW_CSV_PATH)

X = df["text"].astype(str)
y = df["category"].astype(str)

print(f"Total de documentos carregados: {len(df)}")
print(f"Variável preditora X: {X.name}")
print(f"Variável-alvo y:       {y.name} (Classes: {list(y.unique())})")
display(df.head(3))


Total de documentos carregados: 2225
Variável preditora X: text
Variável-alvo y:       category (Classes: ['tech', 'business', 'sport', 'entertainment', 'politics'])


,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...


### 4. Divisão Estratificada de Treino e Teste (80% / 20%)

Aplicamos `train_test_split` com `stratify=y` e `random_state=42`. Isso garante que as proporções observadas na EDA (~23% sport, ~23% business, ~18.7% politics, ~18% tech, ~17.4% entertainment) sejam rigorosamente preservadas em ambos os conjuntos, evitando viés de amostragem.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Conjunto de Treino : {len(X_train)} amostras ({len(X_train) / len(df) * 100:.1f}%)")
print(f"Conjunto de Teste  : {len(X_test)} amostras ({len(X_test) / len(df) * 100:.1f}%)")

# Validação da estratificação
dist_check = pd.DataFrame({
    "Original (%)": (y.value_counts(normalize=True) * 100).round(2),
    "Treino (%)":   (y_train.value_counts(normalize=True) * 100).round(2),
    "Teste (%)":    (y_test.value_counts(normalize=True) * 100).round(2),
})
print("\nVerificação de Proporção das Classes:")
display(dist_check)


Conjunto de Treino : 1780 amostras (80.0%)
Conjunto de Teste  : 445 amostras (20.0%)

Verificação de Proporção das Classes:


,Original (%),Treino (%),Teste (%)
category,,,
business,22.92,22.92,22.92
entertainment,17.35,17.36,17.30
politics,18.74,18.71,18.88
sport,22.97,22.98,22.92
tech,18.02,18.03,17.98


---
## 5. Seleção e Justificativa de Features: Refinamento Progressivo do Vocabulário

Em Text Mining, **cada palavra selecionada para o vocabulário se torna uma feature (coluna da matriz $X$)**.  
A seguir, faremos um estudo de **Seleção de Features em 3 estágios**, demonstrando como a remoção orientada de ruídos e numerais purifica as variáveis preditoras para o classificador Bernoulli.


### Estágio 1: Vetorização Básica com Stopwords Padrão (`stop_words="english"`)
Começamos com a configuração básica do Scikit-Learn e avaliamos quais palavras têm maior presença documental (*Document Frequency*).


In [5]:
vec_estagio1 = CountVectorizer(
    lowercase=True,
    stop_words="english",
    binary=True,
    max_features=5000,
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
)

X_train_e1 = vec_estagio1.fit_transform(X_train)

# Document Frequency das palavras
df_counts_e1 = X_train_e1.sum(axis=0).A1
vocab_e1 = vec_estagio1.get_feature_names_out()

top_e1 = pd.DataFrame({
    "Palavra": vocab_e1,
    "Documentos": df_counts_e1,
    "Proporcao (%)": (df_counts_e1 / len(X_train) * 100).round(2),
}).sort_values(by="Documentos", ascending=False).reset_index(drop=True)

print("Top 10 palavras do Estágio 1 (Apenas 'english'):")
display(top_e1.head(10))


Top 10 palavras do Estágio 1 (Apenas 'english'):


,Palavra,Documentos,Proporcao (%)
0,said,1525,85.67
1,year,977,54.89
2,new,792,44.49
3,time,662,37.19
4,people,635,35.67
5,world,544,30.56
6,years,533,29.94
7,make,526,29.55
8,told,500,28.09
9,just,478,26.85


> **Diagnóstico Crítico do Estágio 1:**  
> Palavras como **`said`** (presente em mais de 70% dos textos!), **`year`**, **`people`**, **`new`**, **`would`** e **`told`** ocupam o topo. Elas são jargões editoriais comuns do jornalismo diário da BBC e não possuem poder discriminativo entre as 5 classes.


### Estágio 2: 1º Refinamento — Removendo Ruídos Editoriais e Verbos de Citação
Expandimos as *stopwords* do NLTK com os termos editoriais identificados no Estágio 1.


In [6]:
nltk_stopwords = set(stopwords.words("english"))

# 1º Acréscimo: termos editoriais e verbos genéricos do jornalismo
ruidos_editoriais = {"said", "mr", "year", "people", "would", "new",
                      "also", "could","told", "just", "still", "may",
                        "world", "bbc", "way", "like", "says", }
stopwords_estagio2 = list(nltk_stopwords.union(ruidos_editoriais))

vec_estagio2 = CountVectorizer(
    lowercase=True,
    stop_words=stopwords_estagio2,
    binary=True,
    max_features=5000,
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
)

X_train_e2 = vec_estagio2.fit_transform(X_train)

df_counts_e2 = X_train_e2.sum(axis=0).A1
vocab_e2 = vec_estagio2.get_feature_names_out()

top_e2 = pd.DataFrame({
    "Palavra": vocab_e2,
    "Documentos": df_counts_e2,
    "Proporcao (%)": (df_counts_e2 / len(X_train) * 100).round(2),
}).sort_values(by="Documentos", ascending=False).reset_index(drop=True)

print("Top 10 palavras do Estágio 2 (Após remover ruídos editoriais):")
display(top_e2.head(10))


Top 10 palavras do Estágio 2 (Após remover ruídos editoriais):


,Palavra,Documentos,Proporcao (%)
0,one,856,48.09
1,last,754,42.36
2,two,691,38.82
3,time,662,37.19
4,first,635,35.67
5,years,533,29.94
6,make,526,29.55
7,made,511,28.71
8,three,479,26.91
9,back,467,26.24


> **Diagnóstico Crítico do Estágio 2:**  
> Ao removermos *said* e *mr*, um novo fenômeno ficou evidente: **numerais por extenso** como **`one`** (48%), **`last`** (42%) e **`two`** (38%) subiram imediatamente para as primeiras posições!  
> Isso ocorre devido à regra do Manual de Redação da BBC de escrever números de 1 a 9 por extenso. Esses numerais ocorrem amplamente em todas as categorias (*"two goals"*, *"two ministers"*, *"two companies"*), tendo variância inter-classes quase nula.


### Estágio 3: 2º Refinamento — Removendo Numerais, Ruídos de Passagem e Unindo Stopwords do Scikit-Learn
Eliminamos numerais por extenso, verbos de passagem (*going*, *set*, *say*, *end*) e adjetivos neutros (*good*).  
Além disso, realizamos a **união formal e explícita** com o conjunto `ENGLISH_STOP_WORDS` do Scikit-Learn (318 palavras), evitando que conectivos como *however*, *part* e *see* vazem para o vocabulário.


In [7]:
# 2º Acréscimo: numerais cardinais/ordinais, termos temporais e verbos/adjetivos neutros de passagem
numerais_e_neutros = {
    "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
    "first", "second", "third", "last", "next",
    "told", "added", "make", "made", "get", "take", "well", "back", "years", "time",
    "much", "many", "since", "week",
    # Verbos de passagem, elocução e adjetivos/substantivos neutros
    "going", "good", "set", "say", "end", "number",
}

# União formal e explícita: Scikit-Learn (318) + NLTK (179) + Ruídos Editoriais + Numerais/Neutros
stopwords_definitivas = list(
    set(ENGLISH_STOP_WORDS)
    .union(nltk_stopwords)
    .union(ruidos_editoriais)
    .union(numerais_e_neutros)
)

vec_definitivo = CountVectorizer(
    lowercase=True,
    stop_words=stopwords_definitivas,
    binary=True,
    max_features=5000,
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
)

X_train_e3 = vec_definitivo.fit_transform(X_train)

df_counts_e3 = X_train_e3.sum(axis=0).A1
vocab_e3 = vec_definitivo.get_feature_names_out()

top_e3 = pd.DataFrame({
    "Palavra": vocab_e3,
    "Documentos": df_counts_e3,
    "Proporcao (%)": (df_counts_e3 / len(X_train) * 100).round(2),
}).sort_values(by="Documentos", ascending=False).reset_index(drop=True)

print("Top 10 palavras do Estágio 3 (Vocabulário Refinado):")
display(top_e3.head(10))


Top 10 palavras do Estágio 3 (Vocabulário Refinado):


,Palavra,Documentos,Proporcao (%)
0,government,382,21.46
1,work,324,18.20
2,expected,317,17.81
3,company,315,17.70
4,think,313,17.58
5,want,309,17.36
6,old,306,17.19
7,home,300,16.85
8,months,298,16.74
9,firm,297,16.69


### 5.4 Tabela Comparativa: Evolução da Seleção de Features
Comparamos lado a lado o ranking de termos dos 3 estágios, evidenciando o efeito da seleção de features:


In [8]:
comparativo_features = pd.DataFrame({
    "1. Stopwords Padrão ('english')": top_e1["Palavra"].head(10),
    "2. Sem Ruídos Editoriais":        top_e2["Palavra"].head(10),
    "3. Sem Numerais/Tempo (Final)":   top_e3["Palavra"].head(10),
})

print("=" * 75)
print(" EVOLUÇÃO DAS FEATURES MAIS FREQUENTES AO LONGO DO REFINAMENTO")
print("=" * 75)
display(comparativo_features)


 EVOLUÇÃO DAS FEATURES MAIS FREQUENTES AO LONGO DO REFINAMENTO


,1. Stopwords Padrão ('english'),2. Sem Ruídos Editoriais,3. Sem Numerais/Tempo (Final)
0,said,one,government
1,year,last,work
2,new,two,expected
3,time,time,company
4,people,first,think
5,world,years,want
6,years,make,old
7,make,made,home
8,told,three,months
9,just,back,firm


---
## 6. Vetorização Oficial e Diagnóstico de Sanidade Bernoulli

Com o vocabulário devidamente selecionado e justificado, fixamos o vetorizador oficial.  
Aplicamos `fit_transform` exclusivamente no **Treino** e apenas `transform` no **Teste** (prevenindo *data leakage*).


In [9]:
# Vetorizador oficial definitivo
vectorizer = vec_definitivo

# 1. Ajuste e transformação no Treino
X_train_vec = X_train_e3

# 2. Apenas transformação no Teste (sem fit!)
X_test_vec = vectorizer.transform(X_test)

print(f"Matriz de Treino: {X_train_vec.shape[0]} documentos x {X_train_vec.shape[1]} features")
print(f"Matriz de Teste : {X_test_vec.shape[0]} documentos x {X_test_vec.shape[1]} features")


Matriz de Treino: 1780 documentos x 5000 features
Matriz de Teste : 445 documentos x 5000 features


In [10]:
# Diagnóstico estatístico e teste formal de binarização
vocab_size = len(vectorizer.vocabulary_)
n_samples, n_features = X_train_vec.shape
sparsity = 100.0 * (1.0 - (X_train_vec.nnz / (n_samples * n_features)))

print("=" * 70)
print(" DIAGNÓSTICO DA VETORIZAÇÃO BINÁRIA (BERNOULLI)")
print("=" * 70)
print(f"Número de documentos de treino : {n_samples}")
print(f"Tamanho do vocabulário V       : {vocab_size} features")
print(f"Número de entradas não-nulas   : {X_train_vec.nnz}")
print(f"Esparsidade da matriz          : {sparsity:.2f}% (porcentagem de zeros)")

# Teste formal de binarização
valores_unicos = np.unique(X_train_vec.data)
print(f"Valores distintos encontrados  : {valores_unicos}")
assert list(valores_unicos) == [1], "ERRO: Valores não-binários encontrados na matriz!"
print("✔ Sanidade confirmada: Todos os valores não-nulos são estritamente 1 (Booleano).")
print("=" * 70)


 DIAGNÓSTICO DA VETORIZAÇÃO BINÁRIA (BERNOULLI)
Número de documentos de treino : 1780
Tamanho do vocabulário V       : 5000 features
Número de entradas não-nulas   : 186171
Esparsidade da matriz          : 97.91% (porcentagem de zeros)
Valores distintos encontrados  : [1]
✔ Sanidade confirmada: Todos os valores não-nulos são estritamente 1 (Booleano).


### 7. Rastreabilidade: Inspecionando um Documento Real vs Vetor Binário
Comprovamos como um artigo individual de treino é projetado no espaço vetorial booleano:


In [11]:
exemplo_idx = 0
texto_original = X_train.iloc[exemplo_idx]
categoria_exemplo = y_train.iloc[exemplo_idx]

# Palavras ativadas no vetor booleano deste documento
vetor_documento = X_train_vec[exemplo_idx].toarray().flatten()
indices_ativos = np.where(vetor_documento == 1)[0]
palavras_ativas = vectorizer.get_feature_names_out()[indices_ativos]

print(f"Artigo de Exemplo (Categoria: {categoria_exemplo.upper()}):")
print("-" * 70)
print(texto_original[:250] + "...\n")
print(f"Total de palavras ativas (x_j = 1) no vocabulário: {len(palavras_ativas)}")
print("Primeiras 25 palavras ativas identificadas:")
print(", ".join(palavras_ativas[:25]) + ", ...")


Artigo de Exemplo (Categoria: BUSINESS):
----------------------------------------------------------------------
share boost for feud-hit reliance the board of indian conglomerate reliance has agreed a share buy-back  to counter the effects of a power struggle in the controlling family.  the buy-back is a victory for chairman mukesh ambani  whose idea it was. h...

Total de palavras ativas (x_j = 1) no vocabulário: 50
Primeiras 25 palavras ativas identificadas:
agreed, based, board, body, bombay, boost, brother, buy, chairman, chamber, company, completely, controlling, counter, current, died, doubt, effects, fall, family, feud, founder, gone, hit, hopes, ...
